# 1. What is an Outlier?

### Concept & Definition
An outlier is an observation that deviates significantly from the remaining data points in a feature distribution.

### Real-World / Business Example
In transaction logs, a $50,000 credit card purchase in a dataset where the average purchase is $45.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("Cleaned_Validated_Data.csv")

# Quick statistical summary to observe potential extreme values
display(df[["Age", "MonthlyCharges", "TenureYears"]].describe())

,Age,MonthlyCharges,TenureYears
count,956.000000,911.000000,1010.000000
mean,43.044979,64.879341,4.115842
std,22.560566,29.550360,3.402536
min,-5.000000,29.850000,0.000000
25%,25.000000,29.850000,1.000000
50%,45.000000,56.950000,3.000000
75%,52.000000,99.990000,8.000000
max,150.000000,105.500000,10.000000


# 2. Outlier vs. Error

### Distinction Framework
- **Data Error:** An incorrect measurement or data entry glitch (e.g., `Age = 350`, `MonthlyCharges = -999`). Must be removed or corrected.
- **Valid Extreme Observation:** A real, accurate, but rare extreme event (e.g., a high-net-worth individual's account balance). Must be handled carefully without arbitrary deletion.

In [2]:
# Identifying explicit errors vs natural extreme values
errors = df[(df["Age"] > 100) | (df["MonthlyCharges"] < 0)]
print(f"Confirmed Measurement Errors Count: {len(errors)}")

Confirmed Measurement Errors Count: 25


# 3. Interquartile Range (IQR) Method

### Mathematical Foundation
$$IQR = Q_3 - Q_1$$
$$\text{Lower Bound} = Q_1 - 1.5 \times IQR$$
$$\text{Upper Bound} = Q_3 + 1.5 \times IQR$$

### When to Use
Non-normally distributed or skewed numerical data.

In [3]:
# IQR Outlier Detection for MonthlyCharges
Q1 = df["MonthlyCharges"].quantile(0.25)
Q3 = df["MonthlyCharges"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

iqr_outliers = df[(df["MonthlyCharges"] < lower_bound) | (df["MonthlyCharges"] > upper_bound)]
print(f"IQR Lower Bound: {lower_bound:.2f}, Upper Bound: {upper_bound:.2f}")
print(f"IQR Outliers Detected: {len(iqr_outliers)}")

IQR Lower Bound: -75.36, Upper Bound: 205.20
IQR Outliers Detected: 0


# 4. Z-Score Method

### Mathematical Foundation
$$Z = \frac{X - \mu}{\sigma}$$
Points with $|Z| > 3$ are typically flagged as outliers.

### When to Use
Strictly for **normally distributed (Gaussian)** features.

In [4]:
from scipy import stats

# Z-score outlier detection on Age
age_clean = df["Age"].dropna()
z_scores = np.abs(stats.zscore(age_clean))
z_outliers = age_clean[z_scores > 3]

print(f"Z-Score Outliers Flagged (|Z| > 3): {len(z_outliers)}")

Z-Score Outliers Flagged (|Z| > 3): 25


# 5. Percentile Method

### Concept & Definition
Flags values falling outside fixed upper and lower percentiles (e.g., bottom 1% and top 99%).

In [5]:
# Percentile Method
lower_p = df["MonthlyCharges"].quantile(0.01)
upper_p = df["MonthlyCharges"].quantile(0.99)

percentile_outliers = df[(df["MonthlyCharges"] < lower_p) | (df["MonthlyCharges"] > upper_p)]
print(f"1st Percentile: {lower_p:.2f}, 99th Percentile: {upper_p:.2f}")
print(f"Percentile Outliers Flagged: {len(percentile_outliers)}")

1st Percentile: 29.85, 99th Percentile: 105.50
Percentile Outliers Flagged: 0


# 6. Winsorization

### Concept & Definition
Replaces extreme values with values at specified percentiles (e.g., top 5% values are replaced with the 95th percentile value) instead of trimming them.

In [6]:
from scipy.stats.mstats import winsorize

# Winsorize MonthlyCharges at 5% tails
df_winsorized = df.copy()
df_winsorized["MonthlyCharges_Winsorized"] = winsorize(df_winsorized["MonthlyCharges"].fillna(df_winsorized["MonthlyCharges"].median()), limits=[0.05, 0.05])

print("Original Max:", df["MonthlyCharges"].max())
print("Winsorized Max:", df_winsorized["MonthlyCharges_Winsorized"].max())

Original Max: 105.5
Winsorized Max: 105.5


# 7. Capping & Flooring

### Concept & Definition
Clamps values above an upper threshold to the threshold (Capping) and values below a lower threshold to the floor (Flooring).

In [7]:
# Manual Capping and Flooring using IQR bounds
df_capped = df.copy()

df_capped["MonthlyCharges_Capped"] = np.where(
    df_capped["MonthlyCharges"] > upper_bound, upper_bound,
    np.where(df_capped["MonthlyCharges"] < lower_bound, lower_bound, df_capped["MonthlyCharges"])
)

print(f"Capped MonthlyCharges Min: {df_capped['MonthlyCharges_Capped'].min():.2f}")
print(f"Capped MonthlyCharges Max: {df_capped['MonthlyCharges_Capped'].max():.2f}")

Capped MonthlyCharges Min: 29.85
Capped MonthlyCharges Max: 105.50


# 8. Feature Transformation

### Concept & Definition
Applies non-linear mathematical transformations (e.g., $\log(x + 1)$ or $\sqrt{x}$) to compress right-skewed distributions and pull extreme tail values closer to the mean.

In [8]:
# Log Transformation to handle skewness and compress outliers
df_transformed = df.copy()
df_transformed["MonthlyCharges_Log"] = np.log1p(df_transformed["MonthlyCharges"])

print("Original Skewness:", df["MonthlyCharges"].skew())
print("Log-Transformed Skewness:", df_transformed["MonthlyCharges_Log"].skew())

Original Skewness: 0.2088926303035584
Log-Transformed Skewness: -0.16606336774689567


# 9. Treatment Decision Framework

| Scenario | Decision | Rationale |
| :--- | :--- | :--- |
| **Confirmed Data Error** | **Remove** | Corrupts gradient updates and model math. |
| **Valid Natural Extreme** | **Retain / Cap / Transform** | Represents real operational reality; removing it loses domain information. |
| **Tree-Based Models (Random Forest/XGBoost)** | **Retain** | Invariant to monotonic feature scales and extreme values. |
| **Linear Models / Neural Networks** | **Cap / Transform** | Highly sensitive to extreme values. |

In [9]:
# Execution of treatment pipeline: Drop errors, Cap valid extremes for linear models
df_final_outliers = df.copy()

# 1. Drop true errors
df_final_outliers = df_final_outliers[df_final_outliers["Age"].between(0, 100)]

# 2. Cap valid continuous extremes
df_final_outliers["MonthlyCharges"] = np.where(
    df_final_outliers["MonthlyCharges"] > upper_bound, upper_bound, df_final_outliers["MonthlyCharges"]
)

print(f"Final Outlier Cleaned Shape: {df_final_outliers.shape}")
print("Notebook 06 execution completed successfully!")

Final Outlier Cleaned Shape: (909, 9)
Notebook 06 execution completed successfully!
